In [3]:
import os
import xarray as xr
import numpy as np
import regionmask
from matplotlib.colors import ListedColormap
import xesmf as xe

In [ ]:
import xarray as xr
# Open 1 km grid SSP population file
SSP1_2050 = xr.open_dataset('/Users/slcohen7/Downloads/mal/KEEP_pop_data_gridded_SSPs/SSP1_total_2050.nc4')['Band1']
# Calculate min/max lat and lon values (to calculate how many grid cells to add together for regridding)
maxy,miny,maxx,minx = SSP1_2050.lat.max(),SSP1_2050.lat.min(),SSP1_2050.lon.max(),SSP1_2050.lon.min(),
#Outputs regridded file at 0.1 * 0.1 degree resolution
#Note: takes several seconds
SSP1_2050 = SSP1_2050[:-50,:].coarsen(lat=int(0.1/((maxy-miny)/len(SSP1_2050.lat[:-1]))),lon=int(0.1/((maxx-minx)/len(SSP1_2050.lon[:-1])))).sum()

In [2]:
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
pop_2050 = xr.open_dataset(f"{POP_DIR}ssp2_total_2050.nc4")["Band1"]

In [3]:
maxy, miny, maxx, minx = pop_2050.lat.max(), pop_2050.lat.min(), pop_2050.lon.max(), pop_2050.lon.min()

In [4]:
SSP2_2050 = pop_2050[:-50, :].coarsen(lat=int(0.1/((maxy-miny)/len(pop_2050.lat[:-1]))), lon=int(0.1/((maxx-minx)/len(pop_2050.lon[:-1])))).sum()

In [18]:
pop_2050[:, :].coarsen(lat=int(0.1/((maxy-miny)/len(pop_2050.lat[:-1]))), lon=int(0.1/((maxx-minx)/len(pop_2050.lon[:-1])))).sum()

ValueError: Could not coarsen a dimension of size 16730 with window 12 and boundary='exact'. Try a different 'boundary' option.

In [ ]:
cmap = ListedColormap(["grey", "lemonchiffon", "lightgreen", "yellowgreen", "lightseagreen", "steelblue", "navy", "k"])

In [13]:
lon=int(0.1/((maxx-minx)/len(pop_2050.lon[:-1])))

In [16]:
pop_2050[:-50, :]

<xarray.DataArray 'Band1' (lat: 16680, lon: 43200)> Size: 3GB
[720576000 values with dtype=float32]
Coordinates:
  * lat      (lat) float64 133kB -55.77 -55.76 -55.75 ... 83.2 83.21 83.22
  * lon      (lon) float64 346kB -180.0 -180.0 -180.0 ... 180.0 180.0 180.0
Attributes:
    grid_mapping:     crs
    SourceBandIndex:  0
    long_name:        GDAL Band Number 1

In [20]:
new_lon

<xarray.DataArray 'lon' (lon: 3600)> Size: 29kB
array([-179.95, -179.85, -179.75, ...,  179.75,  179.85,  179.95])
Coordinates:
  * lon      (lon) float64 29kB -179.9 -179.8 -179.8 ... 179.8 179.8 179.9
Attributes:
    units:    degrees_east

In [ ]:
SSP2_2050.plot(levels=[0, 1, 25, 100, 250, 1000, 5000, 100000, 7241105], cmap=cmap)

In [7]:
OBS_DIR = "/glade/work/awells/air_quality/O3_obs/"
o3_obs = xr.open_dataset(f"{OBS_DIR}Delang_BME_OSDMA8_1990_2017.nc")["ozone"]
obs_base = o3_obs.rename({'longitude': 'lon', 'latitude': 'lat'})

# Define the higher-resolution grid to match observations (0.1 x 0.1)
new_lat = obs_base['lat']
new_lon = obs_base['lon']

In [ ]:
new_grid_with_bounds = {'lon': new_lon['lon'].values,
                        'lat': np.linspace(-55.77, 83.64, 641, endpoint=True),
                        'lon_b': np.linspace(-179.95-0.1/2, 179.95+0.1/2, 3601, endpoint=True),
                        'lat_b': np.linspace(-59.95-0.1/2, 74.95+0.1/2, 1351, endpoint=True),
                       }

pop_grid_with_bounds = {'lon': pop_2050['lon'].values,
                        'lat': pop_2050['lat'].values,
                        'lon_b': np.linspace(-179.995833-0.00833333/2, 179.995833+0.00833333/2, 43201, endpoint=True),
                        'lat_b': np.linspace(-55.77083-0.00833333/2, 83.6375+0.00833333/2, 16731, endpoint=True),
                       }

In [ ]:
83.3+55.77/0.1

In [ ]:
np.linspace(-55.77, 83.3, 641, endpoint=True)

In [ ]:
regridder_conserve = xe.Regridder(pop_grid_with_bounds, new_grid_with_bounds, method='conservative')

In [ ]:
dr_conserve = regridder_conserve(pop_2050)
dr_conserve

In [ ]:
POP_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"
OBS_DIR = "/glade/work/awells/air_quality/O3_obs/"
SAVE_DIR = "/glade/work/awells/air_quality/SSP_pop/SSP2/"

pop_2050 = xr.open_dataset(f"{POP_DIR}ssp2_total_2050.nc4")["Band1"]
o3_obs = xr.open_dataset(f"{OBS_DIR}Delang_BME_OSDMA8_1990_2017.nc")["ozone"]

In [ ]:
nested_grid_with_bounds = {'lon': np.linspace(-140, -40, 161),
                           'lat': np.linspace(10, 70, 121),
                           'lon_b': np.linspace(-140-0.625/2, -40+0.625/2, 162),
                           'lat_b': np.linspace(10-0.5/2, 70+0.5/2, 122),
                          }

In [ ]:
obs_base = o3_obs.rename({'longitude': 'lon', 'latitude': 'lat'})

# Define the higher-resolution grid to match observations (0.1 x 0.1)
new_lat = obs_base['lat']
new_lon = obs_base['lon']

# Interpolate to the new grid
new_pop = pop_2050.interp(lat=new_lat, lon=new_lon, method='linear')


out_file = "ssp2_total_regrid_2050.nc"
out_path = os.path.join(SAVE_DIR, out_file)

print(f"Saving to {out_path}")
new_pop.to_netcdf(out_path)


In [ ]:
cmap = ListedColormap(["grey", "lemonchiffon", "lightgreen", "yellowgreen", "lightseagreen", "steelblue", "navy", "k"])

In [ ]:
pop_2050.plot(levels=[0, 1, 25, 100, 250, 1000, 5000, 100000, 7241105], cmap=cmap)

In [ ]:
ds = xr.open_dataset("/glade/work/awells/air_quality/SSP_pop/ssp2_total_2050.nc4")

In [ ]:
ds["Band1"].plot()